### STAT 238 Final Project 

- Student: Stephen Lu
- Topic: Normalizing Flows as Learned Proposal Distributions for Metropolis-Hastings Sampling 

For my final project, I will be implementing the Adaptive Monte Carlo augmented with normalizing flows algorithm proposed by [Gabrié et al. (2022)](https://arxiv.org/abs/2105.12603). The main idea is to use augment local MCMC sampling (ex: MALA) with non-local transition kernels parameterized by a normalizing flow trained via maximum-likelihood on the samples generated by the local MCMC. The hope is that the non-local transitions will allow the sampler to escape local modes and explore the target distribution more efficiently. In this notebook, I test my implementation on a molecular conformation sampling problem where the target distribution is the Boltzmann distribution of Alanine dipeptide, a small molecule with 22 atoms commonly used as a benchmark in molecular simulation.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float32

# a context tensor to send data to the right device and dtype via '.to(ctx)'
ctx = torch.zeros([], device=device, dtype=dtype)

# a brief check if this module is the main executable (or imported)
main = __name__ == "__main__"

print(f"Running on device: {device}, dtype: {dtype}")

Running on device: cpu, dtype: torch.float32


### Load the MD trajectories and the potential energy function

In [10]:
import os
from pathlib import Path
from bgmol.datasets import Ala2TSF300
from bgflow.distribution.energy.openmm import OpenMMEnergy

root = Path().resolve().parent / "data"
is_data_here = os.path.isfile(root / "Ala2TSF300.npy")
assert root.is_dir(), f"Data directory {root} does not exist. Please create it and place the dataset there."

# Load the MD trajectory, xyz coordinates, temperature, and energy function for the alanine dipeptide system.
dataset = Ala2TSF300(root=str(root), download=(not is_data_here), read=True)

dim = dataset.dim
system = dataset.system
coordinates = dataset.coordinates
temperature = dataset.temperature
target_energy: OpenMMEnergy = dataset.get_energy_model(n_workers=1)

print(system)
print(coordinates.shape)

Using downloaded and verified file: /var/folders/12/28m4qkt55h91b56d_g_y0q6w0000gq/T/alanine-dipeptide-nowater.pdb
system:
  identifier: AlanineDipeptideTSF
  parameters: {}

(1000000, 22, 3)


### Define the Internal Coordinate Transform

The raw alanine dipeptide MD data contains 1 million frames of 66-dimensional Cartesian coordinates (22 atoms x 3 dimensions). Naively parameterizing the normalizing flow in this Cartesian space will yield poor results because of the high dimensionality and the redundant degrees of freedom from translations and rotations of the molecule. Instead, we follow the protocol established in [Noé et al. (2019)](https://arxiv.org/abs/1812.01729) by transforming the raw coordinates into a mixed coordinate system consisting of whitened backbone coordinates and normalized internal coordinates. This transformation is invertible and has a tractable Jacobian determinant, making it suitable for use in normalizing flows all while reducing the effective dimensionality of the problem.

In [ ]:
import bgflow as bg

# throw away 6 degrees of freedom (rotation and translation)
dim_cartesian = len(system.rigid_block) * 3 - 6
dim_bonds = len(system.z_matrix)
dim_angles = dim_bonds
dim_torsions = dim_bonds

# create the coordinate transformer that performs x -> z
all_data = torch.from_numpy(coordinates).to(ctx).reshape(-1, dim)
coordinate_transform = bg.MixedCoordinateTransformation(
    data=all_data,
    z_matrix=system.z_matrix,
    fixed_atoms=system.rigid_block,
    keepdims=dim_cartesian,
    normalize_angles=True,
).to(ctx)

# example forward transformation x -> z
bonds, angles, torsions, cartesian, dlogp = coordinate_transform.forward(all_data[:3])
bonds.shape, angles.shape, torsions.shape, cartesian.shape, dlogp.shape

(torch.Size([3, 17]),
 torch.Size([3, 17]),
 torch.Size([3, 17]),
 torch.Size([3, 9]),
 torch.Size([3, 1]))

### Pick de-correlated samples to initialize the local MCMC chains

